# Session 5 <br> Sequence Tagging

In the previous sessions we have been using models to assign a single class to entire texts. Now we will focus on a more fine grained task: sequence tagging (or classifying every individual token of the text).

There are multiple types of sequence tagging task, for example, _Part of Speech_ (POS) or _Named Entitiy Recognition_ (NER) tagging.

When ussing Transformers, the main difference between both tasks is the classification head at the end of the encoder.

## Before Training

We will need to set up some thing before we start training our models.

### Install and Update libraries

In [ ]:
! pip install transformers -U --quiet
! pip install datasets -U --quiet
! pip install scikit-learn -U --quiet
! pip install tqdm -U --quiet

### Import libraries

In [1]:
import numpy as np
import torch
from transformers import AutoTokenizer, DataCollatorForTokenClassification, TrainingArguments, AutoModelForTokenClassification, Trainer
from datasets import load_dataset
from sklearn.metrics import classification_report
from tqdm.notebook import tqdm

### Evaluating Performance

We will also need a function to compute the performance of our models.
We will use scikit-learn classification report for that.

In [2]:
def compute_metrics(eval_pred):
    temp_predictions, temp_labels = eval_pred
    temp_predictions = np.argmax(temp_predictions, axis=2)
    temp_predictions = temp_predictions.flatten()
    temp_labels = temp_labels.flatten()

    predictions = []
    labels = []
    for p, l in zip(temp_predictions, temp_labels):
        if l != -100:
            predictions.append(p)
            labels.append(l)

    report = classification_report(labels, predictions, output_dict=True, zero_division=0.0)
    for k in list(report.keys()):
        if isinstance(report[k], dict):
            for vk in report[k]:
                report[f'{k}_{vk}'] = report[k][vk]
            del report[k]
    return report

### Get the Tokenizer

We will need a tokenizer to turn the text from string to integer tokens.

In [3]:
# Select Base Model
base_model_path = 'google-bert/bert-base-cased'

# Load Tokenizers
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

### Prepare Data

We also need to prepare our Training, Validation, and Test Data.

First we need to upload the dataset to our virutal machine in colab, then we need to read the content of the files, after that we need to create the Train, Validation, and Test split, and finally we will apply the tokenizer to the dataset.

In [4]:
# Download the datset
ds = load_dataset('DFKI-SLT/few-nerd', 'supervised')

# Take a smaller subset of the data
ds['train'] = ds['train'].take(2**12)
ds['validation'] = ds['validation'].take(2**8)
ds['test'] = ds['test'].take(2**10)

# Define a function to tokenize text
def preprocess_function(example):
    output = tokenizer(example['tokens'], is_split_into_words=True)
    aligned_labels = [0 if i is None else example['ner_tags'][i] for i in output.word_ids()]
    output['labels'] = aligned_labels
    return output

# Tokenize dataset
tokenized_ds = ds.map(preprocess_function).remove_columns(['id', 'tokens', 'ner_tags', 'fine_ner_tags'])

# Prepare data collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

README.md:   0%|          | 0.00/7.13k [00:00<?, ?B/s]

few-nerd.py:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/16.9M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/2.43M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/4.84M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/18824 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/37648 [00:00<?, ? examples/s]

Map:   0%|          | 0/4096 [00:00<?, ? examples/s]

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

Map:   0%|          | 0/1024 [00:00<?, ? examples/s]

### Set Training Arguments

We will use the same training arguments on all experiments for comparability.

In [9]:
# Set training arguments
training_args = TrainingArguments(
    eval_strategy = 'steps',
    logging_steps = 64,
    eval_steps = 64,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 16,
    num_train_epochs = 1,
    report_to = []
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [11]:
from transformers import BertForSequenceClassification

clf = AutoModelForTokenClassification.from_pretrained("google-bert/bert-base-cased", num_labels=9)


Some weights of BertForTokenClassification were not initialized from the model checkpoint at google-bert/bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Experiment

In [12]:
"""
WRITE HERE YOUR CODE TO LOAD AND TRAIN THE MODEL FOR TOKEN CLASSIFICATION
"""

trainer = Trainer(
    model=clf,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    # callbacks=[fft_memory_logger],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Run Training
trainer.train()

# Move the model to CPU and empty the GPU cache to avoid interfeering with the meassurement of future models
clf.to('cpu')
torch.cuda.empty_cache()

Step,Training Loss,Validation Loss,Accuracy,0 Precision,0 Recall,0 F1-score,0 Support,1 Precision,1 Recall,1 F1-score,1 Support,2 Precision,2 Recall,2 F1-score,2 Support,3 Precision,3 Recall,3 F1-score,3 Support,4 Precision,4 Recall,4 F1-score,4 Support,5 Precision,5 Recall,5 F1-score,5 Support,6 Precision,6 Recall,6 F1-score,6 Support,7 Precision,7 Recall,7 F1-score,7 Support,8 Precision,8 Recall,8 F1-score,8 Support,Macro avg Precision,Macro avg Recall,Macro avg F1-score,Macro avg Support,Weighted avg Precision,Weighted avg Recall,Weighted avg F1-score,Weighted avg Support
64,0.673700,0.339944,0.897073,0.961453,0.977671,0.969494,6225.000000,0.722581,0.533333,0.613699,210.000000,0.438596,0.146199,0.219298,171.000000,1.000000,0.035088,0.067797,57.000000,0.770624,0.864560,0.814894,443.000000,0.530120,0.812933,0.641750,433.000000,0.750000,0.383721,0.507692,258.000000,0.852292,0.956190,0.901257,525.000000,0.877193,0.458716,0.602410,218.000000,0.766984,0.574268,0.593143,8540.000000,0.898349,0.897073,0.887551,8540.000000
128,0.330000,0.283403,0.908080,0.979224,0.969157,0.974164,6225.000000,0.838462,0.519048,0.641176,210.000000,0.535948,0.479532,0.506173,171.000000,0.518519,0.491228,0.504505,57.000000,0.858427,0.862302,0.860360,443.000000,0.653846,0.745958,0.696872,433.000000,0.502825,0.689922,0.581699,258.000000,0.902970,0.868571,0.885437,525.000000,0.672131,0.752294,0.709957,218.000000,0.718039,0.708668,0.706705,8540.000000,0.914129,0.908080,0.909452,8540.000000
192,0.282800,0.260629,0.923653,0.961551,0.980241,0.970806,6225.000000,0.835366,0.652381,0.732620,210.000000,0.618056,0.520468,0.565079,171.000000,0.648148,0.614035,0.630631,57.000000,0.914948,0.801354,0.854392,443.000000,0.738683,0.829099,0.781284,433.000000,0.712000,0.689922,0.700787,258.000000,0.940358,0.900952,0.920233,525.000000,0.780488,0.733945,0.756501,218.000000,0.794400,0.746933,0.768037,8540.000000,0.922297,0.923653,0.922169,8540.000000
256,0.272200,0.228557,0.930328,0.980338,0.969157,0.974715,6225.000000,0.835556,0.895238,0.864368,210.000000,0.715084,0.748538,0.731429,171.000000,0.590164,0.631579,0.610169,57.000000,0.819444,0.932280,0.872228,443.000000,0.789340,0.718245,0.752116,433.000000,0.629508,0.744186,0.682060,258.000000,0.922932,0.935238,0.929044,525.000000,0.822581,0.701835,0.757426,218.000000,0.789439,0.808477,0.797062,8540.000000,0.932677,0.930328,0.930899,8540.000000
320,0.267300,0.206006,0.937822,0.978885,0.975582,0.977231,6225.000000,0.828947,0.900000,0.863014,210.000000,0.741379,0.754386,0.747826,171.000000,0.622951,0.666667,0.644068,57.000000,0.878261,0.911964,0.894795,443.000000,0.793103,0.796767,0.794931,433.000000,0.718631,0.732558,0.725528,258.000000,0.922495,0.929524,0.925996,525.000000,0.827957,0.706422,0.762376,218.000000,0.812512,0.819319,0.815085,8540.000000,0.938245,0.937822,0.937847,8540.000000
384,0.227700,0.225950,0.934778,0.968814,0.983133,0.975921,6225.000000,0.831776,0.847619,0.839623,210.000000,0.725714,0.742690,0.734104,171.000000,0.804878,0.578947,0.673469,57.000000,0.884187,0.896163,0.890135,443.000000,0.773672,0.773672,0.773672,433.000000,0.704036,0.608527,0.652807,258.000000,0.942418,0.935238,0.938815,525.000000,0.868263,0.665138,0.753247,218.000000,0.833751,0.781236,0.803532,8540.000000,0.933010,0.934778,0.933277,8540.000000
448,0.251200,0.207332,0.939110,0.974948,0.981526,0.978226,6225.000000,0.850000,0.809524,0.829268,210.000000,0.719298,0.719298,0.719298,171.000000,0.637931,0.649123,0.643478,57.000000,0.896703,0.920993,0.908686,443.000000,0.745763,0.812933,0.777901,433.000000,0.843137,0.666667,0.744589,258.000000,0.958250,0.918095,0.937743,525.000000,0.790476,0.761468,0.775701,218.000000,0.824056,0.804403,0.812766,8540.000000,0.939111,0.939110,0.938663,8540.000000
512,0.225800,0.204388,0.936417,0.978548,0.974618,0.976579,6225.000000,0.835681,0.847619,0.841608,210.000000,0.731844,0.766082,0.748571,171.000000,0.487179,0.666667,0.562963,57.000000,0.873118,0.916479,0.894273,443.000000,0.766147,0.794457,0.780045,433.000000,0.7

## Example

In [36]:
label_mapping = {
    0: '',
    1:'art',
    2:'building',
    3:'event',
    4:'location',
    5:'organization',
    6:'other',
    7:'person',
    8:'product'
}

output = clf(torch.tensor([tokenized_ds['test'][0]['input_ids']]).to('cuda:0'))
prediction = output.logits.argmax(dim=2).cpu().detach().numpy()[0]
print(f'{"Token":<15}{"Label":<15}Prediction')
for token, label, pred in zip(tokenized_ds['test'][0]['input_ids'], tokenized_ds['test'][0]['labels'], prediction):
    print(f'{tokenizer.decode(token):<15}{label_mapping[label]:<15}{label_mapping[pred]}')

Token          Label          Prediction
[CLS]                         
In                            
the                           
early                         
1930s                         
the                           
band                          
moved                         
to                            
the                           
G              building       building
##rill         building       building
Room           building       building
of                            building
the                           building
Taft           building       building
Hotel          building       building
in                            
New            location       location
York           location       location
;                             
the                           
band                          
was                           
renamed                       
`                             
`                             
George         organization   organization
Hall   

In [82]:
 # tokenized_ds['test'][0]['labels']

## Test

In [ ]:
"""
WRITE HERE YOUR CODE COMPUTE THE CLASSIFICATION REPORT OF THE ENTIRE TEST SET
"""

In [21]:
clf.to('cuda:0')

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [55]:
from sklearn.metrics import classification_report
from scipy.special import expit  # stable sigmoid

# Step 1: Get predictions
predictions = trainer.predict(tokenized_ds["test"])

# Step 2: Apply sigmoid to logits and threshold

probs = expit(predictions.predictions)  # shape: (N, C)

# Step 3: Binarize predictions using threshold (e.g., 0.5)
y_pred = (probs > 0.5).astype(int)

# Step 4: True labels
y_true = predictions.label_ids  # shape: (N, C)

# Step 5: Optional class names
label_names = [f"Label_{i}" for i in range(y_true.shape[1])]




In [ ]:
label_mapping = {
    0: 'O',
    1:'art',
    2:'building',
    3:'event',
    4:'location',
    5:'organization',
    6:'other',
    7:'person',
    8:'product'
}

In [94]:
def to_bio(labels):
    bio_labels = []
    prev = 0
    for label in labels:
        if int(label) == 0:
            bio_labels.append('O')
        # elif label == -100:
        #     bio_labels.append(-100)
        elif label != prev:
            bio_labels.append(f"B-{label_mapping[int(label)]}")
        else:
            bio_labels.append(f"I-{label_mapping[int(label)]}")
        prev = label
    return bio_labels

# # Example
# tokens = ["Barack", "Obama", "was", "born", "in", "Hawaii", "."]
# labels = ["PER", "PER", "O", "O", "O", "LOC", "O"]

# bio_labels = to_bio(labels)
# print(list(zip(tokens, bio_labels)))


In [73]:
%pip install seqeval

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     |████████████████████████████████| 43 kB 1.1 MB/s eta 0:00:011
Using legacy 'setup.py install' for seqeval, since package 'wheel' is not installed.
    Running setup.py install for seqeval ... done
Note: you may need to restart the kernel to use updated packages.


In [74]:
from seqeval.metrics import classification_report as classification_report_seq

# Example true and predicted labels
y_true = [['B-PER', 'O', 'B-LOC'], ['B-ORG', 'I-ORG']]
y_pred = [['B-PER', 'O', 'B-LOC'], ['B-ORG', 'O']]

print(classification_report_seq(y_true, y_pred))


              precision    recall  f1-score   support

         LOC       1.00      1.00      1.00         1
         ORG       0.00      0.00      0.00         1
         PER       1.00      1.00      1.00         1

   micro avg       0.67      0.67      0.67         3
   macro avg       0.67      0.67      0.67         3
weighted avg       0.67      0.67      0.67         3



In [ ]:
# Step 6: Print classification report
print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

In [27]:
list(label_mapping.values())

['',
 'art',
 'building',
 'event',
 'location',
 'organization',
 'other',
 'person',
 'product']

In [34]:
len(tokenized_ds['test'])

1024

In [44]:
prediction

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 0, 4, 4, 0, 0,
       0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 0, 0, 0, 0])

In [45]:
np.array(tokenized_ds['test'][0]['labels'])

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 0, 0, 2, 2, 0, 4, 4, 0, 0,
       0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 0, 0, 0, 0])

In [83]:
# tokenized_ds['test'][1]['labels']

In [91]:
y_true = []
y_pred = []
for i in range(len(tokenized_ds['test'])):
    output = clf(torch.tensor([tokenized_ds['test'][i]['input_ids']]).to('cuda'))
    prediction = output.logits.argmax(dim=2).cpu().detach().numpy()[0]
    y_true.append(tokenized_ds['test'][i]['labels'])
    y_pred.append(list(map(int, prediction)))

In [99]:
clf.to('cpu')

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [98]:
torch.cuda.empty_cache()

In [95]:
y_true_l = list(map(to_bio, y_true))
y_pred_l = list(map(to_bio, y_pred))

In [97]:
print(classification_report_seq(y_true_l, y_pred_l))

              precision    recall  f1-score   support

         art       0.72      0.69      0.70       121
    building       0.48      0.59      0.53       133
       event       0.40      0.54      0.46        94
    location       0.77      0.83      0.80       853
organization       0.61      0.70      0.65       565
       other       0.56      0.60      0.58       269
      person       0.85      0.91      0.87       552
     product       0.49      0.63      0.55       130

   micro avg       0.68      0.76      0.72      2717
   macro avg       0.61      0.69      0.64      2717
weighted avg       0.69      0.76      0.72      2717

